In [1]:
import os
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition\\prototype'

In [2]:
os.chdir('../')
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition'

In [4]:
from datetime import datetime, timezone
from typing import List, Dict
import pandas as pd
import requests
from include.helpers import read_yaml
from include.constants import CONFIG_FILE_PATH
import json

In [5]:
CONFIG_FILE_PATH

WindowsPath('config/config.yaml')

In [6]:
USGS_URL = read_yaml(CONFIG_FILE_PATH).data_ingestion.source_URL
USGS_URL

[2026-03-10 02:25:15,994: INFO: helpers: yaml file: config\config.yaml loaded successfully:]


'https://earthquake.usgs.gov/fdsnws/event/1/query'

In [7]:
USGS_URL = read_yaml(CONFIG_FILE_PATH).data_ingestion.source_URL

def fetch_raw_usgs_window(
    start_time: datetime,
    end_time: datetime,
    min_magnitude: float = 2.5,
) -> List[Dict]:
    """
    Fetch one time window of earthquake data from USGS and return
    a flattened list of dictionaries.
    """
    params = {
        "format": "geojson",
        "starttime": start_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "endtime": end_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "minmagnitude": min_magnitude,
        "orderby": "time-asc",
        "limit": 20000,
    }

    response = requests.get(USGS_URL, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    return data

[2026-03-10 02:25:16,075: INFO: helpers: yaml file: config\config.yaml loaded successfully:]


In [11]:
start_time = datetime(2020, 1, 2, tzinfo=timezone.utc)
end_time = datetime(2020, 1, 4, tzinfo=timezone.utc)

data = fetch_raw_usgs_window(
    start_time=start_time,
    end_time=end_time,
    min_magnitude=2.5,
)

os.makedirs("prototype/samples", exist_ok=True)
output_path = f"prototype/samples/usgs_{start_time.strftime('%Y_%m_%d')}_to_{end_time.strftime('%Y_%m_%d')}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Exported JSON to: {output_path}")

Exported JSON to: prototype/samples/usgs_2020_01_02_to_2020_01_04.json
